# Task 2 — Theme Extraction

This notebook identifies complaint themes in reviews using keyword matching and TF-IDF analysis.

**Input:** `data/processed/reviews_with_sentiment.csv`  
**Output:** `data/processed/reviews_with_themes.csv`

**Themes:**
- Account Access Issues (login, password, sign in)
- OTP & Security Issues (otp, code, verification)
- Transaction Performance (slow, delay, transfer, payment)
- App Stability Issues (crash, bug, error)
- UI & UX Issues (ui, design, interface, navigation)
- Feature Requests (feature, fingerprint, update)
- Other

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.theme_extraction import load_data, clean_text, extract_keywords, analyze_themes, get_top_themes, save_output
from src.config import PathConfig

paths = PathConfig()
print(f'Input: {paths.sentiment_output}')
print(f'Output: {paths.themes_output}')

In [ ]:
# Load sentiment-enriched data
df = load_data(paths.sentiment_output)
print(f'Loaded {len(df):,} reviews')
df[['bank', 'review', 'sentiment_label', 'sentiment_score']].head()

In [ ]:
# TF-IDF keyword discovery
df_clean = clean_text(df)
keywords = extract_keywords(df_clean)
print('Top 30 corpus keywords (TF-IDF):')
print(', '.join(keywords[:30]))

In [ ]:
# Assign themes
df_clean = analyze_themes(df_clean)

print('\nTop 5 themes overall:')
print(get_top_themes(df_clean).to_string())

In [ ]:
# Theme distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall theme distribution
theme_counts = df_clean['identified_theme'].value_counts()
axes[0].barh(theme_counts.index[::-1], theme_counts.values[::-1], color='#3498db')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_title('Theme Distribution (All Banks)', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Negative reviews only — by bank and theme
neg_df = df_clean[df_clean['sentiment_label'] == 'NEGATIVE']
neg_theme_bank = neg_df.groupby(['bank', 'identified_theme']).size().unstack(fill_value=0)
neg_theme_bank.T.plot(kind='bar', ax=axes[1], colormap='tab10')
axes[1].set_title('Negative Reviews: Theme × Bank', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Theme')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/theme_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-bank complaint breakdown
for bank in ['CBE', 'BOA', 'DASHEN']:
    bank_neg = neg_df[neg_df['bank'] == bank]
    top = bank_neg['identified_theme'].value_counts().head(3)
    total_neg = len(bank_neg)
    total = len(df_clean[df_clean['bank'] == bank])
    pct = round(total_neg / total * 100, 1)
    print(f'\n{bank}: {total_neg} negative reviews ({pct}% of total)')
    for theme, count in top.items():
        print(f'  - {theme}: {count}')

In [ ]:
# Save final output
save_output(df_clean, paths.themes_output)
print(f'\nPipeline complete. Output: {paths.themes_output}')